# Window — Hinge Constraint Engine Demo

This notebook demonstrates the **deterministic constraint reasoning engine** for hinge-to-mounting-plate compatibility selection.

The engine evaluates every hinge + plate combination against 14 constraint rules (brand lock, series compatibility, overlay range, weight capacity, etc.) and returns only provably correct configurations — no LLM inference involved.

**What's covered:**
1. Loading the product catalog (hinges & mounting plates)
2. Exploring the catalog data
3. Running customer scenarios through the constraint engine
4. Visualizing constraint results and failure analysis
5. Interactive "what-if" exploration

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import display, Image, HTML

# Add parent directory so the engine package is importable
sys.path.insert(0, str(Path("..").resolve()))

from engine import (
    ConcealedHinge, MountingPlate, CustomerRequirements,
    HingeConstraintEngine, load_from_json,
    ApplicationType, CabinetType, CabinetPosition, RuleCategory,
)

# Load catalog data using production loader
data_dir = Path("..") / "sample-data"
hinges, plates = load_from_json(data_dir)

# Load raw JSON for image references
with open(data_dir / "hinges.json") as f:
    hinges_raw = {h["sku"]: h for h in json.load(f)}
with open(data_dir / "mounting_plates.json") as f:
    plates_raw = {p["sku"]: p for p in json.load(f)}

def get_image_path(sku):
    """Get the image path for a product SKU, or None."""
    entry = hinges_raw.get(sku) or plates_raw.get(sku)
    if entry and entry.get("image"):
        return data_dir / entry["image"]
    return None

def show_product_images(hinge_sku, plate_sku, width=200):
    """Display hinge and plate images side by side if available."""
    h_path = get_image_path(hinge_sku)
    p_path = get_image_path(plate_sku)
    html = '<div style="display:flex; gap:20px; align-items:flex-start; margin:10px 0;">'
    if h_path and h_path.exists():
        html += f'<div style="text-align:center"><img src="{h_path}" width="{width}"><br><small>{hinge_sku}</small></div>'
    if p_path and p_path.exists():
        html += f'<div style="text-align:center"><img src="{p_path}" width="{width}"><br><small>{plate_sku}</small></div>'
    html += '</div>'
    if h_path or p_path:
        display(HTML(html))

print(f"Loaded {len(hinges)} hinges and {len(plates)} mounting plates")
print(f"Using production engine (Pydantic models, indexed pre-filtering, no R010 derating)")

## 1. Catalog Overview

Let's look at what's in the product catalog — hinges and mounting plates across brands.

In [ ]:
# Hinge catalog summary
print("=" * 80)
print("HINGE CATALOG")
print("=" * 80)
print(f"{'SKU':<22} {'Brand':<8} {'Series':<20} {'App':<14} {'Angle':>5}  {'Weight':>6}  {'SC':>3}  {'Price':>7}")
print("-" * 80)
for h in hinges:
    sc = "Yes" if h.soft_close else "No"
    price = f"${h.price_usd:>5.2f}" if h.price_usd is not None else "   N/A"
    print(f"{h.sku:<22} {h.brand:<8} {h.series.value:<20} {h.application.value:<14} {h.opening_angle_deg:>4}\u00b0  {h.max_door_weight_kg:>5.1f}kg  {sc:>3}  {price:>7}")

In [ ]:
# Mounting plate catalog summary
print("=" * 80)
print("MOUNTING PLATE CATALOG")
print("=" * 80)
print(f"{'SKU':<22} {'Brand':<8} {'Series':<10} {'Type':<14} {'Mount':<14} {'Cabinet':<12} {'Price':>7}")
print("-" * 80)
for p in plates:
    price = f"${p.price_usd:>5.2f}" if p.price_usd is not None else "   N/A"
    print(f"{p.sku:<22} {p.brand:<8} {p.series:<10} {p.plate_type.value:<14} {p.mounting_method.value:<14} {p.cabinet_type.value:<12} {price:>7}")

In [ ]:
# Brand and application distribution
from collections import Counter

brand_counts = Counter(h.brand for h in hinges)
app_counts = Counter(h.application.value for h in hinges)
angle_counts = Counter(h.opening_angle_deg for h in hinges)

print("Hinges by brand:", dict(brand_counts))
print("Hinges by application:", dict(app_counts))
print("Hinges by opening angle:", dict(angle_counts))
print()
print("Plates by brand:", dict(Counter(p.brand for p in plates)))
print("Plates by type:", dict(Counter(p.plate_type.value for p in plates)))

## Product Gallery

Reference photos for hinges and mounting plates in the catalog. Products with `null` images (Hafele Duomatic, Grass Nexis, Blum wing/face-frame plates) are not yet available.

In [ ]:
# Display all products that have images
html = '<div style="display:flex; flex-wrap:wrap; gap:24px;">'

for sku, entry in {**hinges_raw, **plates_raw}.items():
    img = entry.get("image")
    if not img:
        continue
    img_path = data_dir / img
    if not img_path.exists():
        continue
    label = f"{entry['brand']} {entry.get('series', '')}"
    kind = "Hinge" if sku in hinges_raw else "Plate"
    html += f'''<div style="text-align:center; max-width:180px;">
        <img src="{img_path}" width="160" style="border:1px solid #ddd; border-radius:4px; padding:4px;">
        <br><small><b>{sku}</b><br>{label}<br><i>{kind}</i></small>
    </div>'''

html += '</div>'
display(HTML(html))

## 2. Constraint Rules

The engine enforces 14 constraint rules. Every hinge + plate pair must pass **all** applicable rules to be a valid configuration.

In [ ]:
# Load and display the constraint rules
with open(data_dir / "compatibility_rules.json") as f:
    rules_data = json.load(f)

print(f"{'ID':<6} {'Name':<30} {'Type':<16} Description")
print("-" * 90)
for rule in rules_data["rules"]:
    print(f"{rule['id']:<6} {rule['name']:<30} {rule['type']:<16} {rule['description']}")

## 3. Running Customer Scenarios

Let's run the five demo scenarios through the engine and examine the results.

In [ ]:
engine = HingeConstraintEngine(hinges, plates)

scenarios = [
    (
        "Scenario 1: Standard kitchen — Blum, full overlay, soft-close",
        CustomerRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
            door_weight_kg=5.2, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, preferred_brand="Blum",
        ),
    ),
    (
        "Scenario 2: Corner cabinet — needs wide-angle hinge",
        CustomerRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=800,
            door_weight_kg=4.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, cabinet_position=CabinetPosition.CORNER,
        ),
    ),
    (
        "Scenario 3: Tall pantry door — 1600mm, heavy, Grass preferred",
        CustomerRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=22, door_height_mm=1600,
            door_weight_kg=14.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, preferred_brand="Grass",
        ),
    ),
    (
        "Scenario 4: Adjacent doors sharing partition — half overlay",
        CustomerRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
            door_weight_kg=4.0, application=ApplicationType.HALF_OVERLAY, desired_overlay_mm=6,
            boring_pattern_mm=45, soft_close=True, preferred_brand="Blum",
            has_adjacent_door=True, adjacent_door_overlay_mm=6,
            partition_thickness_mm=19,
        ),
    ),
    (
        "Scenario 5: CONSTRAINT VIOLATION — heavy corner door",
        CustomerRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=22, door_height_mm=900,
            door_weight_kg=12.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, cabinet_position=CabinetPosition.CORNER,
            preferred_brand="Blum",
        ),
    ),
]

# Store results for later analysis
all_results = []

for title, req in scenarios:
    result = engine.solve_with_explanation(req)
    all_results.append((title, req, result))
    
    print("=" * 75)
    print(f"  {title}")
    print(f"  Status: {result['status']}")
    print(f"  {result['message']}")
    
    if result["status"] == "solved":
        rec = result["recommended"]
        print(f"\n  Recommended: {rec['hinge']['sku']} + {rec['mounting_plate']['sku']}")
        print(f"    {rec['hinge']['description']}")
        price = f"${rec['total_price_per_door_usd']}" if rec['total_price_per_door_usd'] is not None else "N/A"
        print(f"    Hinges/door: {rec['hinges_per_door']}  |  Capacity: {rec['total_weight_capacity_kg']}kg  |  Price: {price}/door")
        show_product_images(rec['hinge']['sku'], rec['mounting_plate']['sku'])
        if result["alternatives"]:
            print(f"    + {len(result['alternatives'])} alternative(s)")
    
    elif result["status"] == "no_solution":
        print(f"\n  Failed rules:")
        for fr in result["failed_rules"]:
            print(f"    [{fr['rule']}] {fr['name']}: {fr['detail']}")
    
    elif result["status"] == "no_solution_for_brand":
        print(f"\n  Alternatives from other brands:")
        for alt in result["alternatives"]:
            price = f"${alt['total_price_per_door_usd']}" if alt['total_price_per_door_usd'] is not None else "N/A"
            print(f"    {alt['hinge']['sku']} — {price}/door")
            show_product_images(alt['hinge']['sku'], alt['mounting_plate']['sku'], width=150)
    
    print()

## 4. Constraint Trace Deep Dive

Let's look at the full constraint trace for Scenario 1 to see how every rule is evaluated.

In [ ]:
# Full constraint trace for Scenario 1's recommended configuration
title, req, result = all_results[0]
print(f"{title}\n")

if result["status"] == "solved":
    trace = result["recommended"]["constraint_trace"]
    print(f"{'Rule':<6} {'Name':<28} {'Result':<6} Detail")
    print("-" * 90)
    for rule in trace:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"{rule['rule']:<6} {rule['name']:<28} {icon:<6} {rule['detail']}")

## 5. Compatibility Matrix

How many valid configurations exist for each hinge × plate combination? This heatmap shows the constraint satisfaction landscape.

In [ ]:
# Build a compatibility matrix: for a "standard" set of requirements,
# how many rules does each hinge+plate pair pass?
baseline_req = CustomerRequirements(
    cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
    door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
    boring_pattern_mm=45, soft_close=False,
)

matrix = []
for h in hinges:
    row = []
    for p in plates:
        config = engine.evaluate(h, p, baseline_req)
        passed = sum(1 for r in config.rule_results if r.passed)
        row.append(passed)
    matrix.append(row)

# Print as a text heatmap
num_rules = len(engine.evaluate(hinges[0], plates[0], baseline_req).rule_results)
print(f"Rules passed (out of {num_rules}) for full_overlay frameless baseline\n")
print(f"{'Hinge SKU':<22}", end="")
for p in plates:
    print(f" {p.sku[-7:]:<8}", end="")
print()
print("-" * (22 + 9 * len(plates)))

for i, h in enumerate(hinges):
    print(f"{h.sku:<22}", end="")
    for j, p in enumerate(plates):
        val = matrix[i][j]
        marker = f" {val}/{num_rules}  " if val < num_rules else " ALL   "
        print(marker, end="")
    print()

## 6. Price vs Capacity Analysis

Comparing valid configurations by price per door and total weight capacity.

In [ ]:
# Find all valid configs for a standard full-overlay requirement (no brand preference)
open_req = CustomerRequirements(
    cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
    door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
    boring_pattern_mm=45, soft_close=False,
)

all_valid = engine.solve(open_req)

def fmt_price(p):
    return f"${p:>9.2f}" if p is not None else "      N/A"

print(f"Found {len(all_valid)} valid configurations for standard full-overlay frameless\n")
print(f"{'Hinge SKU':<22} {'Plate SKU':<22} {'Brand':<8} {'Hinges':>6} {'Capacity':>10} {'Price/Door':>11}")
print("-" * 80)
for config in all_valid:
    print(f"{config.hinge.sku:<22} {config.plate.sku:<22} {config.hinge.brand:<8} "
          f"{config.hinges_per_door:>6} {config.total_weight_capacity_kg:>8.1f}kg "
          f"{fmt_price(config.total_price_usd)}")

# Show images for the cheapest and most capable options
priced = [c for c in all_valid if c.total_price_usd is not None]
if priced:
    print(f"\n--- Best value (cheapest) ---")
    show_product_images(priced[0].hinge.sku, priced[0].plate.sku)
if all_valid:
    most_capable = max(all_valid, key=lambda c: c.total_weight_capacity_kg)
    print(f"--- Highest capacity ({most_capable.total_weight_capacity_kg}kg) ---")
    show_product_images(most_capable.hinge.sku, most_capable.plate.sku)

## 7. Failure Analysis — Why Scenario 5 Has No Solution

When the engine can't find a valid configuration, it identifies the closest match and the specific rules that failed. Let's dig into the constraint violation scenario.

In [ ]:
# Scenario 5: heavy corner door — let's see exactly why it fails
title, req, result = all_results[4]
print(f"{title}\n")
print(f"Requirements:")
print(f"  Cabinet: {req.cabinet_type.value}, Position: {req.cabinet_position.value}")
print(f"  Door: {req.door_thickness_mm}mm thick, {req.door_height_mm}mm tall, {req.door_weight_kg}kg")
print(f"  Application: {req.application.value}, Overlay: {req.desired_overlay_mm}mm")
print(f"  Soft-close: {req.soft_close}, Brand: {req.preferred_brand}")
print()

print(f"Result: {result['status']}")
print(f"{result['message']}\n")

if result.get("closest_match"):
    cm = result["closest_match"]
    print(f"Closest match: {cm['hinge']['sku']} + {cm['mounting_plate']['sku']}")
    print(f"  {cm['hinge']['description']}")
    print(f"  Capacity: {cm['total_weight_capacity_kg']}kg for {cm['hinges_per_door']} hinges")
    print()

if result.get("failed_rules"):
    print("Failed constraints:")
    for fr in result["failed_rules"]:
        print(f"  [{fr['rule']}] {fr['name']}")
        print(f"    {fr['detail']}")

# Show the math behind the weight/angle analysis
num_h = engine.hinges_per_door(req.door_height_mm)
print(f"\n--- Analysis ---")
print(f"Corner cabinet requires >= 155\u00b0 hinge (R013)")
print(f"Door height {req.door_height_mm}mm -> {num_h} hinges (R008)")
print(f"Note: production engine does NOT apply R010 derating")
print(f"Blum 155\u00b0 hinge: 5.0kg max x {num_h} hinges = {5.0 * num_h}kg capacity")
print(f"Door weight: {req.door_weight_kg}kg -> {'FAIL' if req.door_weight_kg > 5.0 * num_h else 'PASS'}")

## 8. Interactive Explorer

Modify the requirements below to test your own scenarios against the constraint engine.

In [ ]:
# === MODIFY THESE VALUES TO TEST YOUR OWN SCENARIOS ===

my_requirements = CustomerRequirements(
    cabinet_type=CabinetType.FRAMELESS,           # FRAMELESS or FACE_FRAME
    door_thickness_mm=19,                          # typical: 16-24mm
    door_height_mm=720,                            # mm (affects hinge count)
    door_weight_kg=5.0,                            # kg
    application=ApplicationType.FULL_OVERLAY,      # FULL_OVERLAY, HALF_OVERLAY, INSET, OVERLAY
    desired_overlay_mm=16,                         # mm
    drilling_distance_mm=5.0,                      # mm (NEW — for exact overlay calc)
    boring_pattern_mm=45,                          # mm (standard: 45)
    soft_close=True,                               # True or False
    cabinet_position=CabinetPosition.STANDARD,     # STANDARD, CORNER, BLIND_CORNER
    preferred_brand=None,                          # "Blum", "Grass", "Hafele", or None
    has_adjacent_door=False,                       # True if sharing a partition
    adjacent_door_overlay_mm=0,                    # mm (only if has_adjacent_door)
    partition_thickness_mm=19,                     # mm (only if has_adjacent_door)
    face_frame_width_mm=0,                         # mm (only if FACE_FRAME cabinet)
)

# === RUN THE ENGINE ===

result = engine.solve_with_explanation(my_requirements)

print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result["status"] == "solved":
    rec = result["recommended"]
    print(f"Recommended configuration:")
    print(f"  Hinge: {rec['hinge']['sku']} — {rec['hinge']['description']}")
    print(f"  Plate: {rec['mounting_plate']['sku']} — {rec['mounting_plate']['description']}")
    print(f"  Hinges per door: {rec['hinges_per_door']}")
    print(f"  Weight capacity: {rec['total_weight_capacity_kg']}kg")
    price = f"${rec['total_price_per_door_usd']}" if rec['total_price_per_door_usd'] is not None else "N/A"
    print(f"  Price per door: {price}")
    show_product_images(rec['hinge']['sku'], rec['mounting_plate']['sku'], width=250)
    print(f"\n  Constraint trace:")
    for rule in rec["constraint_trace"]:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"    [{icon}] {rule['rule']} {rule['name']}: {rule['detail']}")
    
    if result["alternatives"]:
        print(f"\n  Alternatives ({len(result['alternatives'])}):")
        for alt in result["alternatives"]:
            alt_price = f"${alt['total_price_per_door_usd']}" if alt['total_price_per_door_usd'] is not None else "N/A"
            print(f"    {alt['hinge']['sku']} + {alt['mounting_plate']['sku']} — {alt_price}/door")

elif result["status"] == "no_solution":
    if result.get("closest_match"):
        cm = result["closest_match"]
        print(f"Closest match: {cm['hinge']['sku']} + {cm['mounting_plate']['sku']}")
    print(f"\nFailed rules:")
    for fr in result["failed_rules"]:
        print(f"  [{fr['rule']}] {fr['name']}: {fr['detail']}")

elif result["status"] == "no_solution_for_brand":
    for alt in result["alternatives"]:
        alt_price = f"${alt['total_price_per_door_usd']}" if alt['total_price_per_door_usd'] is not None else "N/A"
        print(f"  {alt['hinge']['sku']} — {alt['hinge']['description']} ({alt_price}/door)")
        show_product_images(alt['hinge']['sku'], alt['mounting_plate']['sku'], width=180)